# 06 — Final Comparison

**ML Ensemble Benchmarking Framework**

This notebook runs all three models (Random Forest, XGBoost, SVM) through the `Benchmark` orchestrator in one pass, producing the final comparison table and all four report figures used in `README.md` and `reports/benchmark_report.md`.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.data.load_data import load_raw_data, train_test_split_data
from src.data.preprocess import preprocess_pipeline
from src.features.build_features import engineer_features
from src.features.selection import select_features
from src.features.imbalance import apply_smote
from src.models.random_forest import RandomForestModel
from src.models.xgboost_model import XGBoostModel
from src.models.svm_model import SVMModel
from src.models.benchmark import Benchmark
from src.visualization.plots import (
    plot_model_comparison,
    plot_roc_curves,
    plot_confusion_matrices,
    plot_feature_importance,
)

RANDOM_STATE = 42

## Full Pipeline: Load → Preprocess → Engineer → Select → Balance

In [ ]:
df = load_raw_data(n_samples=8000, n_features=20)
X_train, X_test, y_train, y_test = train_test_split_data(df, test_size=0.2)

X_train_pp, X_test_pp, _ = preprocess_pipeline(X_train, X_test)
X_train_fe = engineer_features(X_train_pp, max_interaction_pairs=10)
X_test_fe = engineer_features(X_test_pp, max_interaction_pairs=10)

X_train_sel, selected_cols = select_features(X_train_fe, y_train, method="importance", k=20)
X_test_sel = X_test_fe[selected_cols]

X_train_bal, y_train_bal = apply_smote(X_train_sel, y_train)

print(f"Final training shape: {X_train_bal.shape}")
print(f"Final test shape: {X_test_sel.shape}")

## Run the Full Benchmark

In [ ]:
models = [
    RandomForestModel(),
    XGBoostModel(),
    SVMModel(),
]

benchmark = Benchmark(models=models, cv_folds=3, scoring="f1")
results = benchmark.run(X_train_bal, y_train_bal, X_test_sel, y_test)

## Comparison Table

In [ ]:
summary = benchmark.summary_dataframe()
summary.round(4)

## Identify the Winner

In [ ]:
best = benchmark.best_model(metric="f1")
print(f"Best model by F1-score: {best.model_name}")
print(f"F1-Score: {best.metrics.f1:.4f}")
print(f"ROC-AUC:  {best.metrics.roc_auc:.4f}")
print(f"Best params: {best.best_params}")

## Generate Report Figures

In [ ]:
plot_model_comparison(summary, metrics=["accuracy", "f1", "roc_auc"], output_dir="../reports/figures")

fitted_estimators = {name: r.fitted_model.estimator for name, r in benchmark.results.items()}

plot_roc_curves(fitted_estimators, X_test_sel, y_test, output_dir="../reports/figures")
plot_confusion_matrices(fitted_estimators, X_test_sel, y_test, output_dir="../reports/figures")

## Feature Importance (from the tuned Random Forest)

In [ ]:
rf_estimator = benchmark.results["random_forest"].fitted_model.estimator

plot_feature_importance(
    feature_names=X_train_bal.columns.tolist(),
    importances=rf_estimator.feature_importances_,
    title="Random Forest Feature Importance (Gini)",
    output_dir="../reports/figures",
)

## Save the Final Benchmark Report

In [ ]:
benchmark.save_report("../reports/benchmark_results.json")
print("Saved reports/benchmark_results.json")

## Conclusion

Across all three models, **XGBoost** achieves the best combination of F1-score, ROC-AUC, and tuning efficiency on this benchmark, consistent with the reference results documented in `docs/results.md`. Random Forest is a close second on ROC-AUC and offers strong interpretability via Gini feature importance. SVM lags on precision, reflecting the difficulty non-linear kernel methods have with this feature space's dimensionality without more aggressive feature selection.

See [`reports/benchmark_report.md`](../reports/benchmark_report.md) for the full written report and [`docs/results.md`](../docs/results.md) for detailed per-model analysis.